In [2]:
import pybamm
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go


v_nom = 3.63 # V
energy_capacity = 18.2 # Wh
nom_capacity = energy_capacity / v_nom # A

c_rates_charge = np.arange(0.1, 0.8, 0.1) 
c_rates_discharge = np.arange(0.1, 1.6, 0.2)

2024-05-21 15:25:11.659 - [WARNING] citations.read_citations(76): Citations could not be read because the 'pybtex' library is not installed. Install 'pybamm[cite]' to enable citation reading.


In [87]:
n_c = []
n_c_at_vnom = []
solutions_charge = []
for i in c_rates_charge:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {i}C until 4.2V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    solutions_charge.append(sol)
    new_n_c = 1 - sol["Current [A]"].data * (sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_c.append(new_n_c)
    n_c_at_vnom.append(new_n_c[np.argmin(np.abs(sol["Terminal voltage [V]"].data - v_nom))])
n_d = []
n_d_at_vnom = []
solutions_discharge = []
for i in c_rates_discharge:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {i}C until 2.5V",
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    solutions_discharge.append(sol)
    new_n_d = 1 - sol["Current [A]"].data * (-sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_d.append(new_n_d)
    n_d_at_vnom.append(new_n_d[np.argmin(np.abs(sol["Terminal voltage [V]"].data - v_nom))])

In [88]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03)
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend2={"y": 0.5, "yanchor": "top"},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for i in range(len(n_c)):
    fig.add_trace(
        go.Scatter(x=solutions_charge[i]["Terminal voltage [V]"].data[1:], y=n_c[i][1:], name=f"{c_rates_charge[i]:.1f}C", legend="legend", mode="lines"), row=1, col=1 
    )
fig.add_vline(x=v_nom)
fig.add_trace(go.Scatter(x=[v_nom], y=[np.mean(n_c_at_vnom)], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=1, col=1)
fig.update_yaxes(row=1, col=1, title_text="$\\text{Charging efficiency } \\eta_c$")
for i in range(len(n_d)):
    fig.add_trace(
        go.Scatter(x=solutions_discharge[i]["Terminal voltage [V]"].data[1:], y=n_d[i][1:], name=f"-{c_rates_discharge[i]:.1f}C", legend="legend2", mode="lines"), row=2, col=1 
    )
fig.add_vline(x=v_nom)
fig.add_trace(go.Scatter(x=[v_nom], y=[np.mean(n_d_at_vnom)], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=2, col=1)
fig.update_xaxes(row=2, col=1, title="$\\text{Terminal Voltage } V$", ticksuffix="V")
fig.update_yaxes(row=2, col=1, title_text="$\\text{Discharging efficiency } \\eta_d$")
fig.write_image("../thesis/Images/charging_efficiency.pdf", "pdf")
print(np.mean(n_c_at_vnom))
print(np.mean(n_d_at_vnom))

0.9790538841002994
1.0432857283283954


In [3]:
experiment = pybamm.Experiment(
    [
        (
            "Charge at 1W until 4.2V",
        )
    ],
    period="1 second"
)
model = pybamm.lithium_ion.SPM()
sim = pybamm.Simulation(model=model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
sol = sim.solve(initial_soc=0)
ocv = sol['Surface open-circuit voltage [V]'].data
soc = np.linspace(0.0, 1.0, ocv.size)


last_charge_soc = []
for i in c_rates_charge:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {i}C until 4.2V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    value = sol['Surface open-circuit voltage [V]'].data[-1]
    idx = np.searchsorted(ocv, value, side='right')
    x1, x2 = ocv[idx - 1], ocv[idx]
    y1, y2 = soc[idx - 1], soc[idx]
    last_charge_soc.append(y1 + (value - x1) * (y2 - y1) / (x2 - x1))


experiment = pybamm.Experiment(
    [
        (
            "Discharge at 1W until 2.5V",
        )
    ],
    period="1 second"
)
model = pybamm.lithium_ion.SPM()
sim = pybamm.Simulation(model=model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
sol = sim.solve(initial_soc=1)
ocv = sol['Surface open-circuit voltage [V]'].data
soc = np.linspace(1.0, 0.0, ocv.size)

last_discharge_soc = []
for i in c_rates_discharge:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {i}C until 2.5V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    value = sol['Surface open-circuit voltage [V]'].data[-1]
    idx = np.searchsorted(ocv, value, side='right')
    x1, x2 = ocv[idx - 1], ocv[idx]
    y1, y2 = soc[idx - 1], soc[idx]
    last_discharge_soc.append(y1 + (value - x1) * (y2 - y1) / (x2 - x1))


In [7]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1, specs=[[{"secondary_y": True}], [{"secondary_y": True}]])
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend2={"y": 0.5, "yanchor": "top"},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)

discharge_capacity = np.array(last_discharge_soc) * energy_capacity # Wh

u_lower = np.sum(c_rates_discharge*np.array(last_discharge_soc)) / np.sum(c_rates_discharge**2)
v_lower = 0
print(u_lower, v_lower)

fig.add_trace(
    go.Scatter(x=c_rates_discharge, y=np.array(last_discharge_soc) * 100, showlegend=False, line_color="red"), row=1, col=1
)
fig.add_trace(
    go.Scatter(x=c_rates_discharge, y=discharge_capacity, showlegend=False, line_color="red"), secondary_y=True, row=1, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(0.0, 1.6, 0.1), y=(u_lower*np.arange(0.0, 1.6, 0.1) + v_lower) * 100, showlegend=False, mode="lines", line_color="red", line_dash="dash"), row=1, col=1
)
fig.update_yaxes(row=1, col=1, title_text="$\\text{Lower State-of-Charge Limit}$", ticksuffix="%", range=(0, 100 / 8))
fig.update_yaxes(row=1, col=1, title_text="$\\text{Lower Energy Content Limit}$", ticksuffix="Wh", secondary_y=True, range=[0, energy_capacity / 8])
fig.update_xaxes(row=1, col=1, title="$\\text{Discharge Rate}$", ticksuffix="C")


charge_capacity = np.array(last_charge_soc) * energy_capacity #Wh

v_upper = 1
u_upper = np.sum(c_rates_charge*(np.array(last_charge_soc) - 1)) / np.sum(c_rates_charge**2)
print(u_upper, v_upper)

fig.add_trace(
    go.Scatter(x=c_rates_charge, y=np.array(last_charge_soc) * 100, showlegend=False, line_color="green"), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=c_rates_charge, y=charge_capacity, showlegend=False, line_color="green"), secondary_y=True, row=2, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(0, 0.8, 0.1), y=(u_upper*np.arange(0, 0.8, 0.1) + v_upper) * 100, showlegend=False, mode="lines", line_color="green", line_dash="dash"), row=2, col=1
)
fig.update_yaxes(row=2, col=1, title_text="$\\text{Upper State-of-Charge Limit}$", ticksuffix="%", range=(100 / 14 * 13, 100))
fig.update_yaxes(row=2, col=1, title_text="$\\text{Upper Energy Content Limit}$", ticksuffix="Wh", secondary_y=True, range=(energy_capacity / 14 * 13, energy_capacity))
fig.update_xaxes(row=2, col=1, title="$\\text{Charge Rate}$", ticksuffix="C")
fig.write_image("../thesis/Images/energy_limits.pdf", "pdf")

0.06903789991796308 0
-0.0818502720355245 1


In [10]:
print(np.mean(1 - c_rates_charge * nom_capacity * 0.030 / 3.63))
print(np.mean(1 + c_rates_discharge * nom_capacity * 0.030 / 3.63))

0.9834255401498074
1.0331489197003847
